# S&A Fee and Enrollment Context

This notebook documents and validates student Services & Activities (S&A) fee rates and Seattle Central College enrollment context for the S&A Budget Explorer.

## Coverage

- FY2023–24
- FY2024–25
- FY2025–26
- FY2026–27 — retained as current/future context

The main dashboard currently uses FY2023–24 through FY2025–26.

## Research Questions

1. How much does a full-time student pay in S&A fees at different credit loads?
2. What is the 12-credit minimum full-time fee?
3. What is the typical 15-credit fee?
4. At what credit level does the S&A fee reach its maximum?
5. How have S&A fee rates changed across academic years?
6. How many students does Seattle Central serve each academic year?
7. How should enrollment figures be interpreted alongside S&A budget allocations?

## Planned Workflow

1. Load manually verified S&A fee data
2. Validate academic years and credit levels
3. Check for missing values and duplicates
4. Validate the S&A fee cap
5. Summarize 12-credit, 15-credit, and maximum fees
6. Calculate year-over-year changes
7. Load Seattle Central enrollment context
8. Validate enrollment measures and sources
9. Create clean research summary tables
10. Export processed context data
11. Document limitations

In [1]:
from pathlib import Path

import pandas as pd


# =========================================================
# PATHS
# =========================================================

PROJECT_ROOT = Path.cwd().parent

SA_FEE_PATH = (
    PROJECT_ROOT
    / "data"
    / "manual"
    / "sa_fee_rates.csv"
)


# =========================================================
# LOAD DATA
# =========================================================

sa_fees = pd.read_csv(
    SA_FEE_PATH
)


print("S&A FEE DATASET")
print("=" * 50)

print(
    f"Rows: {len(sa_fees):,}"
)

print(
    f"Columns: {len(sa_fees.columns):,}"
)

print(
    "\nAcademic years:"
)

print(
    sa_fees[
        "academic_year"
    ]
    .value_counts()
    .sort_index()
)


display(
    sa_fees.head()
)

S&A FEE DATASET
Rows: 40
Columns: 9

Academic years:
academic_year
2023-24    10
2024-25    10
2025-26    10
2026-27    10
Name: count, dtype: int64


,academic_year,credits,quarterly_sa_fee,full_time_flag,fee_cap_flag,dashboard_status,source_document,source_pdf_page,notes
0,2023-24,12,141.82,True,False,current,2023-24_tuition_fee_schedule.pdf,2,NaN
1,2023-24,13,149.18,True,False,current,2023-24_tuition_fee_schedule.pdf,2,NaN
2,2023-24,14,156.54,True,False,current,2023-24_tuition_fee_schedule.pdf,2,NaN
3,2023-24,15,163.90,True,False,current,2023-24_tuition_fee_schedule.pdf,2,Typical 15-credit reference
4,2023-24,16,171.26,True,False,current,2023-24_tuition_fee_schedule.pdf,2,NaN


In [2]:
# =========================================================
# VALIDATION
# =========================================================

print("S&A FEE VALIDATION")
print("=" * 50)


# ---------------------------------------------------------
# 1. DUPLICATES
# ---------------------------------------------------------

duplicate_count = (
    sa_fees
    .duplicated(
        subset=[
            "academic_year",
            "credits",
        ]
    )
    .sum()
)

print(
    f"Duplicate year-credit rows: {duplicate_count}"
)


# ---------------------------------------------------------
# 2. REQUIRED VALUES
# ---------------------------------------------------------

required_columns = [
    "academic_year",
    "credits",
    "quarterly_sa_fee",
    "full_time_flag",
    "fee_cap_flag",
    "dashboard_status",
    "source_document",
    "source_pdf_page",
]

missing_required = (
    sa_fees[
        required_columns
    ]
    .isna()
    .sum()
)

print(
    "\nMissing required values:"
)

print(
    missing_required
)


# ---------------------------------------------------------
# 3. CREDIT COVERAGE
# ---------------------------------------------------------

expected_credits = set(
    range(
        12,
        22,
    )
)

credit_validation = []

for year, group in (
    sa_fees
    .groupby(
        "academic_year"
    )
):

    actual_credits = set(
        group[
            "credits"
        ]
    )

    missing_credits = sorted(
        expected_credits
        - actual_credits
    )

    extra_credits = sorted(
        actual_credits
        - expected_credits
    )

    credit_validation.append(
        {
            "academic_year": year,
            "rows": len(group),
            "missing_credits": missing_credits,
            "extra_credits": extra_credits,
        }
    )


credit_validation = pd.DataFrame(
    credit_validation
)

print(
    "\nCredit coverage:"
)

display(
    credit_validation
)


# ---------------------------------------------------------
# 4. FEE CAP VALIDATION
# ---------------------------------------------------------

cap_validation = []

for year, group in (
    sa_fees
    .groupby(
        "academic_year"
    )
):

    group = (
        group
        .sort_values(
            "credits"
        )
    )

    fee_18 = (
        group.loc[
            group[
                "credits"
            ]
            == 18,
            "quarterly_sa_fee",
        ]
        .iloc[0]
    )

    capped_rows = (
        group[
            group[
                "credits"
            ]
            .between(
                18,
                21,
            )
        ]
    )

    cap_matches = (
        capped_rows[
            "quarterly_sa_fee"
        ]
        .eq(
            fee_18
        )
        .all()
    )

    cap_flags_correct = (
        capped_rows[
            "fee_cap_flag"
        ]
        .eq(
            True
        )
        .all()
    )

    cap_validation.append(
        {
            "academic_year": year,
            "18_credit_fee": fee_18,
            "18_to_21_same_fee": cap_matches,
            "cap_flags_correct": cap_flags_correct,
        }
    )


cap_validation = pd.DataFrame(
    cap_validation
)

print(
    "\nFee-cap validation:"
)

display(
    cap_validation
)


# ---------------------------------------------------------
# FINAL RESULT
# ---------------------------------------------------------

validation_passed = (
    duplicate_count == 0
    and missing_required.sum() == 0
    and credit_validation[
        "missing_credits"
    ]
    .apply(
        len
    )
    .sum()
    == 0
    and credit_validation[
        "extra_credits"
    ]
    .apply(
        len
    )
    .sum()
    == 0
    and cap_validation[
        "18_to_21_same_fee"
    ]
    .all()
    and cap_validation[
        "cap_flags_correct"
    ]
    .all()
)


print(
    "\n"
    + "=" * 50
)

print(
    "VALIDATION PASSED"
    if validation_passed
    else "VALIDATION FAILED"
)

S&A FEE VALIDATION
Duplicate year-credit rows: 0

Missing required values:
academic_year       0
credits             0
quarterly_sa_fee    0
full_time_flag      0
fee_cap_flag        0
dashboard_status    0
source_document     0
source_pdf_page     0
dtype: int64

Credit coverage:


,academic_year,rows,missing_credits,extra_credits
0,2023-24,10,[],[]
1,2024-25,10,[],[]
2,2025-26,10,[],[]
3,2026-27,10,[],[]



Fee-cap validation:


,academic_year,18_credit_fee,18_to_21_same_fee,cap_flags_correct
0,2023-24,185.98,True,True
1,2024-25,191.92,True,True
2,2025-26,198.50,True,True
3,2026-27,204.80,True,True



VALIDATION PASSED


In [3]:
# =========================================================
# S&A FEE SUMMARY
# =========================================================

summary_rows = []


for year, group in (
    sa_fees
    .groupby(
        "academic_year"
    )
):

    group = (
        group
        .sort_values(
            "credits"
        )
    )


    fee_12 = (
        group.loc[
            group["credits"] == 12,
            "quarterly_sa_fee",
        ]
        .iloc[0]
    )


    fee_15 = (
        group.loc[
            group["credits"] == 15,
            "quarterly_sa_fee",
        ]
        .iloc[0]
    )


    fee_18 = (
        group.loc[
            group["credits"] == 18,
            "quarterly_sa_fee",
        ]
        .iloc[0]
    )


    summary_rows.append(
        {
            "academic_year": year,
            "full_time_min_credits": 12,
            "full_time_min_fee": fee_12,
            "typical_credits": 15,
            "typical_15_credit_fee": fee_15,
            "fee_cap_credits": 18,
            "max_quarterly_sa_fee": fee_18,
            "full_time_fee_range": (
                f"${fee_12:,.2f}–${fee_18:,.2f}"
            ),
        }
    )


sa_fee_summary = pd.DataFrame(
    summary_rows
)


# =========================================================
# YEAR-OVER-YEAR CHANGE
# =========================================================

sa_fee_summary[
    "typical_fee_yoy_change"
] = (
    sa_fee_summary[
        "typical_15_credit_fee"
    ]
    .pct_change()
    * 100
)


sa_fee_summary[
    "max_fee_yoy_change"
] = (
    sa_fee_summary[
        "max_quarterly_sa_fee"
    ]
    .pct_change()
    * 100
)


print("S&A FEE SUMMARY")
print("=" * 50)

display(
    sa_fee_summary
)


S&A FEE SUMMARY


,academic_year,full_time_min_credits,full_time_min_fee,typical_credits,typical_15_credit_fee,fee_cap_credits,max_quarterly_sa_fee,full_time_fee_range,typical_fee_yoy_change,max_fee_yoy_change
0,2023-24,12,141.82,15,163.90,18,185.98,$141.82–$185.98,NaN,NaN
1,2024-25,12,146.38,15,169.15,18,191.92,$146.38–$191.92,3.203173,3.193892
2,2025-26,12,151.40,15,174.95,18,198.50,$151.40–$198.50,3.428909,3.428512
3,2026-27,12,156.20,15,180.50,18,204.80,$156.20–$204.80,3.172335,3.173804


In [4]:
# =========================================================
# EXPORT S&A FEE SUMMARY
# =========================================================

OUTPUT_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "sa_fee_summary.csv"
)


sa_fee_summary.to_csv(
    OUTPUT_PATH,
    index=False,
)


print(
    f"Saved: {OUTPUT_PATH}"
)

print(
    f"Rows exported: {len(sa_fee_summary):,}"
)

Saved: /Users/mainguyen/GitHub/seattle-central-sa-budget-dashboard/data/processed/sa_fee_summary.csv
Rows exported: 4


In [5]:
# =========================================================
# LOAD ENROLLMENT CONTEXT
# =========================================================

ENROLLMENT_PATH = (
    PROJECT_ROOT
    / "data"
    / "manual"
    / "enrollment_context.csv"
)

enrollment = pd.read_csv(
    ENROLLMENT_PATH
)

print("ENROLLMENT CONTEXT")
print("=" * 50)

print(
    f"Rows: {len(enrollment):,}"
)

print(
    f"Columns: {len(enrollment.columns):,}"
)

display(
    enrollment
)

ENROLLMENT CONTEXT
Rows: 4
Columns: 7


,academic_year,total_students,enrollment_measure,status,source_name,source_label,notes
0,2023-24,12088.0,annual_headcount,verified,Seattle Colleges,Seattle Central College,Annual total student headcount
1,2024-25,12378.0,annual_headcount,verified,SBCTC Enrollment Data Dashboard,Seattle Central/SVI,Annual headcount; matches Seattle Central publ...
2,2025-26,11672.0,annual_headcount,verified,SBCTC Enrollment Data Dashboard,Seattle Central/SVI,Annual headcount
3,2026-27,NaN,annual_headcount,future,NaN,NaN,Annual academic-year headcount not yet available


In [6]:
# =========================================================
# ENROLLMENT VALIDATION
# =========================================================

print("ENROLLMENT VALIDATION")
print("=" * 50)


# ---------------------------------------------------------
# 1. DUPLICATE YEARS
# ---------------------------------------------------------

duplicate_years = (
    enrollment
    .duplicated(
        subset=["academic_year"]
    )
    .sum()
)

print(
    f"Duplicate academic years: {duplicate_years}"
)


# ---------------------------------------------------------
# 2. EXPECTED YEARS
# ---------------------------------------------------------

expected_years = {
    "2023-24",
    "2024-25",
    "2025-26",
    "2026-27",
}

actual_years = set(
    enrollment["academic_year"]
)

missing_years = sorted(
    expected_years - actual_years
)

extra_years = sorted(
    actual_years - expected_years
)

print(
    f"\nMissing years: {missing_years}"
)

print(
    f"Extra years: {extra_years}"
)


# ---------------------------------------------------------
# 3. ENROLLMENT MEASURE
# ---------------------------------------------------------

measure_check = (
    enrollment[
        "enrollment_measure"
    ]
    .eq(
        "annual_headcount"
    )
    .all()
)

print(
    f"\nAll rows use annual_headcount: {measure_check}"
)


# ---------------------------------------------------------
# 4. VERIFIED YEARS HAVE STUDENT COUNTS
# ---------------------------------------------------------

verified_rows = (
    enrollment[
        enrollment["status"] == "verified"
    ]
)

verified_counts_present = (
    verified_rows[
        "total_students"
    ]
    .notna()
    .all()
)

print(
    "Verified years have student counts: "
    f"{verified_counts_present}"
)


# ---------------------------------------------------------
# 5. FUTURE YEAR SHOULD NOT HAVE FINAL COUNT
# ---------------------------------------------------------

future_rows = (
    enrollment[
        enrollment["status"] == "future"
    ]
)

future_counts_blank = (
    future_rows[
        "total_students"
    ]
    .isna()
    .all()
)

print(
    "Future-year counts are blank: "
    f"{future_counts_blank}"
)


# ---------------------------------------------------------
# FINAL RESULT
# ---------------------------------------------------------

validation_passed = (
    duplicate_years == 0
    and len(missing_years) == 0
    and len(extra_years) == 0
    and measure_check
    and verified_counts_present
    and future_counts_blank
)

print(
    "\n"
    + "=" * 50
)

print(
    "VALIDATION PASSED"
    if validation_passed
    else "VALIDATION FAILED"
)

ENROLLMENT VALIDATION
Duplicate academic years: 0

Missing years: []
Extra years: []

All rows use annual_headcount: True
Verified years have student counts: True
Future-year counts are blank: True

VALIDATION PASSED


In [7]:
# =========================================================
# ENROLLMENT TREND SUMMARY
# =========================================================

enrollment_summary = (
    enrollment[
        enrollment["status"] == "verified"
    ]
    .copy()
    .sort_values(
        "academic_year"
    )
)


enrollment_summary[
    "student_change"
] = (
    enrollment_summary[
        "total_students"
    ]
    .diff()
)


enrollment_summary[
    "student_change_pct"
] = (
    enrollment_summary[
        "total_students"
    ]
    .pct_change()
    * 100
)


print("ENROLLMENT TREND SUMMARY")
print("=" * 50)

display(
    enrollment_summary[
        [
            "academic_year",
            "total_students",
            "student_change",
            "student_change_pct",
        ]
    ]
)

ENROLLMENT TREND SUMMARY


,academic_year,total_students,student_change,student_change_pct
0,2023-24,12088.0,NaN,NaN
1,2024-25,12378.0,290.0,2.399073
2,2025-26,11672.0,-706.0,-5.703668


In [8]:
# =========================================================
# FULL-PERIOD ENROLLMENT CHANGE
# =========================================================

start_students = (
    enrollment_summary.loc[
        enrollment_summary["academic_year"] == "2023-24",
        "total_students",
    ]
    .iloc[0]
)

end_students = (
    enrollment_summary.loc[
        enrollment_summary["academic_year"] == "2025-26",
        "total_students",
    ]
    .iloc[0]
)

full_period_change = (
    end_students
    - start_students
)

full_period_change_pct = (
    full_period_change
    / start_students
    * 100
)


print("FULL-PERIOD ENROLLMENT CHANGE")
print("=" * 50)

print(
    f"2023-24 students: {start_students:,.0f}"
)

print(
    f"2025-26 students: {end_students:,.0f}"
)

print(
    f"Net change: {full_period_change:+,.0f}"
)

print(
    f"Percent change: {full_period_change_pct:+.1f}%"
)

FULL-PERIOD ENROLLMENT CHANGE
2023-24 students: 12,088
2025-26 students: 11,672
Net change: -416
Percent change: -3.4%


In [9]:
# =========================================================
# EXPORT ENROLLMENT SUMMARY
# =========================================================

processed_enrollment = (
    enrollment_summary[
        [
            "academic_year",
            "total_students",
            "student_change",
            "student_change_pct",
            "enrollment_measure",
            "source_name",
            "source_label",
            "notes",
        ]
    ]
    .copy()
)


# Clean numeric types for export
processed_enrollment[
    "total_students"
] = (
    processed_enrollment[
        "total_students"
    ]
    .astype("Int64")
)

processed_enrollment[
    "student_change"
] = (
    processed_enrollment[
        "student_change"
    ]
    .round()
    .astype("Int64")
)

processed_enrollment[
    "student_change_pct"
] = (
    processed_enrollment[
        "student_change_pct"
    ]
    .round(2)
)


OUTPUT_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "enrollment_summary.csv"
)


processed_enrollment.to_csv(
    OUTPUT_PATH,
    index=False,
)


print("ENROLLMENT SUMMARY EXPORTED")
print("=" * 50)

print(
    f"Saved: {OUTPUT_PATH}"
)

print(
    f"Rows exported: {len(processed_enrollment):,}"
)

display(
    processed_enrollment
)

ENROLLMENT SUMMARY EXPORTED
Saved: /Users/mainguyen/GitHub/seattle-central-sa-budget-dashboard/data/processed/enrollment_summary.csv
Rows exported: 3


,academic_year,total_students,student_change,student_change_pct,enrollment_measure,source_name,source_label,notes
0,2023-24,12088,<NA>,NaN,annual_headcount,Seattle Colleges,Seattle Central College,Annual total student headcount
1,2024-25,12378,290,2.4,annual_headcount,SBCTC Enrollment Data Dashboard,Seattle Central/SVI,Annual headcount; matches Seattle Central publ...
2,2025-26,11672,-706,-5.7,annual_headcount,SBCTC Enrollment Data Dashboard,Seattle Central/SVI,Annual headcount


In [10]:
# =========================================================
# LOAD CANONICAL BUDGET DATA FOR FINAL CONTEXT SUMMARY
# =========================================================

BUDGET_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "sa_budget_canonical.csv"
)

budget = pd.read_csv(
    BUDGET_PATH
)

print("CANONICAL BUDGET DATA")
print("=" * 50)

print(
    f"Rows: {len(budget):,}"
)

print(
    f"Columns: {len(budget.columns):,}"
)

print("\nColumns:")
print(
    budget.columns.tolist()
)

display(
    budget.head()
)

CANONICAL BUDGET DATA
Rows: 155
Columns: 11

Columns:
['source_document', 'source_page', 'source_url', 'program_name_raw', 'program_name_standardized', 'mapping_type', 'fiscal_year', 'metric_type', 'amount', 'raw_value', 'metric_context']


,source_document,source_page,source_url,program_name_raw,program_name_standardized,mapping_type,fiscal_year,metric_type,amount,raw_value,metric_context
0,2023-24_sa_fee_memo.pdf,3,https://studentleadership.seattlecentral.edu/s...,ASC Book Fund,ASC Book Fund,same,2022-23,allocation,3000.00,"$3,000.00",NaN
1,2023-24_sa_fee_memo.pdf,3,https://studentleadership.seattlecentral.edu/s...,"Equity, Diversity, Inclusion, and Community","Accessibility, Community, & Opportunity",renamed_or_reorganized,2022-23,allocation,11500.00,"$11,500.00",NaN
2,2023-24_sa_fee_memo.pdf,3,https://studentleadership.seattlecentral.edu/s...,Associated Student Council,Associated Student Council,same,2022-23,allocation,56953.00,"$56,953.00",NaN
3,2023-24_sa_fee_memo.pdf,3,https://studentleadership.seattlecentral.edu/s...,Bruce Mckenna Writing Center,Bruce McKenna Writing Center,spelling_variant,2022-23,allocation,56039.01,"$56,039.01",NaN
4,2023-24_sa_fee_memo.pdf,3,https://studentleadership.seattlecentral.edu/s...,Cultural Programming & Development (CAB),Cultural Programming & Development (CAB),same,2022-23,allocation,116767.00,"$116,767.00",NaN


In [11]:
# =========================================================
# BUILD YEAR CONTEXT SUMMARY
# =========================================================

DASHBOARD_YEARS = [
    "2023-24",
    "2024-25",
    "2025-26",
]


# ---------------------------------------------------------
# 1. BUDGET YEAR SUMMARY
# ---------------------------------------------------------

budget_dashboard = (
    budget[
        (budget["fiscal_year"].isin(DASHBOARD_YEARS))
        & (budget["metric_type"] == "allocation")
    ]
    .copy()
)


budget_year_summary = (
    budget_dashboard
    .groupby(
        "fiscal_year",
        as_index=False,
    )
    .agg(
        total_sa_allocation=(
            "amount",
            "sum",
        ),
        funded_programs=(
            "program_name_standardized",
            lambda x: x[
                budget_dashboard.loc[
                    x.index,
                    "amount",
                ] > 0
            ].nunique(),
        ),
    )
    .rename(
        columns={
            "fiscal_year": "academic_year"
        }
    )
)


# ---------------------------------------------------------
# 2. FEE YEAR SUMMARY
# ---------------------------------------------------------

fee_year_summary = (
    sa_fee_summary[
        sa_fee_summary[
            "academic_year"
        ]
        .isin(
            DASHBOARD_YEARS
        )
    ]
    [
        [
            "academic_year",
            "full_time_min_fee",
            "typical_15_credit_fee",
            "max_quarterly_sa_fee",
        ]
    ]
    .copy()
)


# ---------------------------------------------------------
# 3. ENROLLMENT YEAR SUMMARY
# ---------------------------------------------------------

enrollment_year_summary = (
    processed_enrollment[
        processed_enrollment[
            "academic_year"
        ]
        .isin(
            DASHBOARD_YEARS
        )
    ]
    [
        [
            "academic_year",
            "total_students",
        ]
    ]
    .copy()
)


# ---------------------------------------------------------
# 4. COMBINE
# ---------------------------------------------------------

year_context_summary = (
    budget_year_summary
    .merge(
        fee_year_summary,
        on="academic_year",
        how="left",
        validate="one_to_one",
    )
    .merge(
        enrollment_year_summary,
        on="academic_year",
        how="left",
        validate="one_to_one",
    )
    .sort_values(
        "academic_year"
    )
    .reset_index(
        drop=True
    )
)


# ---------------------------------------------------------
# 5. CLEAN TYPES / ROUNDING
# ---------------------------------------------------------

year_context_summary[
    "total_sa_allocation"
] = (
    year_context_summary[
        "total_sa_allocation"
    ]
    .round(2)
)

year_context_summary[
    "total_students"
] = (
    year_context_summary[
        "total_students"
    ]
    .astype("Int64")
)


print("YEAR CONTEXT SUMMARY")
print("=" * 50)

display(
    year_context_summary
)

YEAR CONTEXT SUMMARY


,academic_year,total_sa_allocation,funded_programs,full_time_min_fee,typical_15_credit_fee,max_quarterly_sa_fee,total_students
0,2023-24,1513171.18,18,141.82,163.90,185.98,12088
1,2024-25,1733008.51,19,146.38,169.15,191.92,12378
2,2025-26,1801888.48,20,151.40,174.95,198.50,11672


In [12]:
# =========================================================
# FINAL CROSS-DATASET QA
# =========================================================

print("FINAL CONTEXT DATA VALIDATION")
print("=" * 50)


# ---------------------------------------------------------
# EXPECTED YEARS
# ---------------------------------------------------------

expected_years = set(
    DASHBOARD_YEARS
)

actual_years = set(
    year_context_summary[
        "academic_year"
    ]
)

years_correct = (
    actual_years
    == expected_years
)


# ---------------------------------------------------------
# DUPLICATES
# ---------------------------------------------------------

duplicate_years = (
    year_context_summary
    .duplicated(
        subset=[
            "academic_year"
        ]
    )
    .sum()
)


# ---------------------------------------------------------
# MISSING VALUES
# ---------------------------------------------------------

required_context_columns = [
    "academic_year",
    "total_sa_allocation",
    "funded_programs",
    "full_time_min_fee",
    "typical_15_credit_fee",
    "max_quarterly_sa_fee",
    "total_students",
]

missing_required = (
    year_context_summary[
        required_context_columns
    ]
    .isna()
    .sum()
)


# ---------------------------------------------------------
# LOGICAL CHECKS
# ---------------------------------------------------------

allocation_positive = (
    year_context_summary[
        "total_sa_allocation"
    ]
    .gt(0)
    .all()
)

students_positive = (
    year_context_summary[
        "total_students"
    ]
    .gt(0)
    .all()
)

fee_order_correct = (
    (
        year_context_summary[
            "full_time_min_fee"
        ]
        <=
        year_context_summary[
            "typical_15_credit_fee"
        ]
    )
    &
    (
        year_context_summary[
            "typical_15_credit_fee"
        ]
        <=
        year_context_summary[
            "max_quarterly_sa_fee"
        ]
    )
).all()


# ---------------------------------------------------------
# KNOWN BUDGET TOTAL CHECK
# ---------------------------------------------------------

expected_budget_totals = {
    "2023-24": 1513171.18,
    "2024-25": 1733008.51,
    "2025-26": 1801888.48,
}

budget_total_check = all(
    abs(
        year_context_summary.loc[
            year_context_summary[
                "academic_year"
            ] == year,
            "total_sa_allocation",
        ].iloc[0]
        - expected_total
    )
    < 0.01
    for year, expected_total
    in expected_budget_totals.items()
)


# ---------------------------------------------------------
# FINAL RESULT
# ---------------------------------------------------------

validation_passed = (
    years_correct
    and duplicate_years == 0
    and missing_required.sum() == 0
    and allocation_positive
    and students_positive
    and fee_order_correct
    and budget_total_check
)


print(
    f"Dashboard years correct: {years_correct}"
)

print(
    f"Duplicate years: {duplicate_years}"
)

print(
    "\nMissing required values:"
)

print(
    missing_required
)

print(
    f"\nAllocations positive: {allocation_positive}"
)

print(
    f"Student counts positive: {students_positive}"
)

print(
    f"Fee ordering correct: {fee_order_correct}"
)

print(
    f"Budget totals match canonical values: {budget_total_check}"
)


print(
    "\n"
    + "=" * 50
)

print(
    "ALL CONTEXT DATA VALIDATED"
    if validation_passed
    else "VALIDATION FAILED"
)


# =========================================================
# EXPORT
# =========================================================

if validation_passed:

    OUTPUT_PATH = (
        PROJECT_ROOT
        / "data"
        / "processed"
        / "year_context_summary.csv"
    )

    year_context_summary.to_csv(
        OUTPUT_PATH,
        index=False,
    )

    print(
        f"\nSaved: {OUTPUT_PATH}"
    )

    print(
        f"Rows exported: {len(year_context_summary):,}"
    )

FINAL CONTEXT DATA VALIDATION
Dashboard years correct: True
Duplicate years: 0

Missing required values:
academic_year            0
total_sa_allocation      0
funded_programs          0
full_time_min_fee        0
typical_15_credit_fee    0
max_quarterly_sa_fee     0
total_students           0
dtype: int64

Allocations positive: True
Student counts positive: True
Fee ordering correct: True
Budget totals match canonical values: True

ALL CONTEXT DATA VALIDATED

Saved: /Users/mainguyen/GitHub/seattle-central-sa-budget-dashboard/data/processed/year_context_summary.csv
Rows exported: 3
